# LEVIR-CD · Siamese U-Net uzun eğitim deneyi

Colab'da **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seç. Sonra **1–4** numaralı hücreleri sırayla çalıştır. İlk prova geçmeden tam eğitimi açma.

Varsayılan ayar, Siamese U-Net'i baştan 30 tur eğitir ve önceki 10 turluk koşuyla karşılaştırır.

Nihai test için **1–2. hücreleri** çalıştır, **3. hücrede** `PREPARE_TEST = True` seç, sonra **7. hücreye** geç. Arşivler [resmî LEVIR-CD kaynağından](https://justchenhao.github.io/LEVIR/) gelir. Model kayıtları ve sonuçlar Google Drive'ında kalır.

## 1. GPU ve proje kodu
GPU görünmüyorsa hücre burada durur; veri indirmez.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/apeates/change-detection.git'
REPO_DIR = Path('/content/change-detection')
DATA_ROOT = Path('/content/levir_cd_data')
BATCH_SIZE = 2
SEED = 42
FULL_EPOCHS = 30
LR_SCHEDULE = 'linear'
MODEL = 'siamese_unet'  # Orijinal koşu için 'early_fusion_unet'.
AUGMENT = False
BASELINE_RUN = 'early_fusion_v2_linear'
RUN_NAME = {'early_fusion_unet': f'early_fusion_v2_{LR_SCHEDULE}',
            'siamese_unet': f'siamese_v1_{LR_SCHEDULE}'}[MODEL]
if AUGMENT:
    RUN_NAME += '_aug'
if FULL_EPOCHS != 10:
    BASELINE_RUN = RUN_NAME  # Aynı ayarlarla yapılmış 10 turluk koşu.
    RUN_NAME += f'_e{FULL_EPOCHS}'
COMPARE_WITH_BASELINE = RUN_NAME != BASELINE_RUN
BENCHMARK_ARCHITECTURES = MODEL == 'siamese_unet' and FULL_EPOCHS == 10

if not shutil.which('nvidia-smi') or subprocess.run(['nvidia-smi', '-L'], check=False).returncode != 0:
    raise RuntimeError('Colab çalışma zamanı ayarından GPU seçip bu hücreyi tekrar çalıştır.')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    '--branch', 'main', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'main'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print(f'Git commit: {REVISION}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO_DIR / 'requirements-train.txt'), 'gdown>=5.2,<7'], check=True)
subprocess.run([sys.executable, '-c',
                "import torch; assert torch.cuda.is_available(), 'PyTorch GPU erişimi yok'; print(torch.cuda.get_device_name(0))"],
               check=True)
sys.path.insert(0, str(REPO_DIR))

## 2. Eğitim kayıtları için Drive'ı bağla
Colab izin isterse kendi Google hesabında onayla.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
RUNS_ROOT = Path('/content/drive/MyDrive/change-detection-runs')
RUN_ROOT = RUNS_ROOT / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Normalde None kalsın. Resmî indirme çalışmazsa iki ZIP'i içeren kendi Drive klasörünü yaz.
DRIVE_ARCHIVES = None  # Örnek: Path('/content/drive/MyDrive/levir-cd')
S2LOOKING_ARCHIVE = Path('/content/drive/MyDrive/S2Looking.zip')

def get_s2looking_archive():
    import zipfile

    archive = S2LOOKING_ARCHIVE if S2LOOKING_ARCHIVE.is_file() else Path('/content/S2Looking.zip')
    if archive == Path('/content/S2Looking.zip') and not zipfile.is_zipfile(archive):
        import gdown
        from gdown.exceptions import FileURLRetrievalError

        print('Yayıncının S2Looking arşivi indiriliyor. Arşiv yaklaşık 10 GB.')
        try:
            gdown.download(id='1YAUrUp3QtZ6VMM1nSuqOjveIVO9ADOY1', output=str(archive))
        except FileURLRetrievalError as error:
            raise RuntimeError('Google Drive, yayıncının arşivi için indirme bağlantısı vermedi. '
                               'Daha sonra yeniden dene veya https://github.com/S2Looking/Dataset '
                               'sayfasındaki Baidu Drive bağlantısından ZIP’i indir. '
                               'Arşivi MyDrive/S2Looking.zip olarak koyup hücreyi tekrar çalıştır.') from error
    if not zipfile.is_zipfile(archive):
        raise ValueError(f'Geçerli S2Looking ZIP arşivi bulunamadı: {archive}')
    return archive

print(f'Eğitim kayıtları: {RUN_ROOT}')
if COMPARE_WITH_BASELINE:
    print(f'Doğrulama referansı: {BASELINE_RUN}')

## 3. Veriyi hazırla
Eğitim için varsayılan ayarlar **445 train** ve **64 val** çifti hazırlar. 6. hücredeki doğrulama için `PREPARE_TRAIN = False` seç. 7. hücredeki nihai test için `PREPARE_TEST = True` seç. Bu mod **128 test** çiftini hazırlar.

In [ ]:
import tempfile
import zipfile
from pathlib import PurePosixPath

import gdown
from levir_data import list_samples

PREPARE_TEST = False  # Nihai test için True seç.
PREPARE_TRAIN = True  # Yalnızca doğrulama için False seç.
SPLITS = {
    'train': ('18GuoCuBn48oZKAlEo-LrNwABrFhVALU-', 445),
    'val': ('1BqSt4ueO7XAyQ_84mUjswUSJt13ZBuzG', 64),
}
if PREPARE_TEST:
    SPLITS = {'test': ('1jj3qJD_grJlgIhUWO09zibRGJe0R4Tn0', 128)}
ARCHIVE_CACHE = Path('/content/levir_cd_archives')

def verify_split(directory, expected):
    count = len(list_samples(directory))
    if count != expected:
        raise ValueError(f'{directory.name}: {count} eşleşme bulundu, {expected} bekleniyor. ZIP adını kontrol et.')
    return count

def get_archive(split, file_id):
    if DRIVE_ARCHIVES is not None:
        archive = Path(DRIVE_ARCHIVES) / f'{split}.zip'
    else:
        ARCHIVE_CACHE.mkdir(parents=True, exist_ok=True)
        archive = ARCHIVE_CACHE / f'{split}.zip'
        if not zipfile.is_zipfile(archive):
            downloaded = gdown.download(id=file_id, output=str(archive), quiet=False)
            if not downloaded:
                raise RuntimeError(f'{split}.zip indirilemedi. Hata mesajını kontrol et veya DRIVE_ARCHIVES kullan.')
    if not zipfile.is_zipfile(archive):
        raise ValueError(f'Geçerli ZIP bulunamadı: {archive}')
    return archive

def extract_split(archive, destination, expected):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=destination.parent) as directory:
        staged = Path(directory) / destination.name
        for folder in ('A', 'B', 'label'):
            (staged / folder).mkdir(parents=True)
        with zipfile.ZipFile(archive) as source:
            for entry in source.infolist():
                member = PurePosixPath(entry.filename)
                folder = member.parent.name
                if entry.is_dir() or member.suffix.lower() != '.png' or member.name.startswith('.'):
                    continue
                if folder not in ('A', 'B', 'label'):
                    continue
                target = staged / folder / member.name
                with source.open(entry) as incoming, target.open('xb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
        verify_split(staged, expected)
        staged.replace(destination)

for split, (file_id, expected) in SPLITS.items():
    if split == 'train' and not PREPARE_TRAIN:
        continue
    destination = DATA_ROOT / split
    if not destination.exists():
        extract_split(get_archive(split, file_id), destination, expected)
    print(f'{split}: {verify_split(destination, expected)} eşleşen görüntü çifti')

## 4. Kısa GPU provası
Dört eğitim parçası, bir doğrulama görüntüsü, iki tur. Bu aşamadaki skor model başarısı olarak kullanılmaz. Çıktıda `Last:` ve `Best:` yollarını görmelisin.

In [ ]:
import json
from importlib.metadata import version

import torch

def run_training(name, epochs, smoke_tiles=None):
    output = RUN_ROOT / name
    output.mkdir(parents=True, exist_ok=True)
    revision_file = output / 'git_revision.txt'
    if revision_file.exists() and revision_file.read_text().strip() != REVISION:
        raise RuntimeError('Bu kayıt başka bir Git commit’inden. Yeni bir RUN_NAME seç.')
    revision_file.write_text(REVISION + '\n')
    command = [sys.executable, '-u', 'train_unet.py', '--device', 'cuda',
               '--data-root', str(DATA_ROOT), '--output-dir', str(output),
               '--epochs', str(epochs), '--batch-size', str(BATCH_SIZE),
               '--lr-schedule', LR_SCHEDULE, '--seed', str(SEED), '--model', MODEL]
    if AUGMENT:
        command.append('--augment')
    if smoke_tiles is not None:
        command.extend(['--smoke-tiles', str(smoke_tiles)])
    if (output / 'last.pt').is_file():
        command.append('--resume')
    runtime = {'python': sys.version.split()[0], 'torch': torch.__version__,
               'numpy': version('numpy'), 'pillow': version('Pillow'),
               'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)}
    runtime_file = output / 'runtime.json'
    if not runtime_file.exists():
        runtime_file.write_text(json.dumps(runtime, indent=2) + '\n', encoding='utf-8')
    subprocess.run(command, cwd=REPO_DIR, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})

run_training('smoke', epochs=2, smoke_tiles=4)

## 5. Tam eğitim — prova başarılı olduktan sonra
Prova başarılıysa `RUN_FULL = True` yap. Uzun koşu başlangıçtan eğitilir. Öğrenme hızı `0.001`'den `0.0001`'e `FULL_EPOCHS` tur boyunca doğrusal iner. Eğitim kesilirse aynı koşu adı, kod sürümü ve `FULL_EPOCHS` ile **1–4** hücrelerini tekrar çalıştır, sonra bu hücreyi aç. Bu koşunun `last.pt` kaydı varsa son tamamlanan turdan devam eder.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    smoke = RUN_ROOT / 'smoke'
    if not all((smoke / name).is_file() for name in ('last.pt', 'best.pt', 'git_revision.txt')):
        raise RuntimeError('Önce 4. hücredeki GPU provasını tamamla.')
    if (smoke / 'git_revision.txt').read_text().strip() != REVISION:
        raise RuntimeError('Bu kod sürümüyle prova yapmak için yeni bir RUN_NAME seç.')
    if torch.load(smoke / 'last.pt', map_location='cpu', weights_only=True)['epoch'] < 2:
        raise RuntimeError('Kısa prova henüz iki turu tamamlamadı; 4. hücreyi tekrar çalıştır.')
    run_training('full', epochs=FULL_EPOCHS)
else:
    print('Tam eğitim kapalı. Önce kısa GPU provasının çıktısını kontrol et.')

## 6. Eşikleri karşılaştır ve doğrulama hatalarını incele
Tam koşu bittikten sonra bu hücreyi çalıştır. Kayıtlı bir koşuyu yeni oturumda değerlendireceksen 1–3. hücreleri çalıştır ve 3. hücrede `PREPARE_TRAIN = False` seç. `full/best.pt` ile yedi eşik ölçülür. 30 ve 10 turluk Siamese koşuları, ortak `0.35` eşiğinde ve her koşunun seçilen doğrulama eşiğinde karşılaştırılır.

In [ ]:
import csv
import json
from IPython.display import Image, display
from evaluate_unet import benchmark_models, compare_validation
from train_unet import best_row

checkpoint = RUN_ROOT / 'full' / 'best.pt'
analysis_dir = RUN_ROOT / 'val_sweep_t035'
baseline_root = RUNS_ROOT / BASELINE_RUN
baseline_dir = baseline_root / 'val_sweep_t035'
baseline_checkpoint = baseline_root / 'full' / 'best.pt'
if not checkpoint.is_file():
    raise FileNotFoundError(f'Tamamlanan koşunun model kaydı bulunamadı: {checkpoint}')
if COMPARE_WITH_BASELINE:
    import torch

    last = checkpoint.with_name('last.pt')
    saved = torch.load(last, map_location='cpu', weights_only=True) if last.is_file() else None
    if saved is None or saved['epoch'] != FULL_EPOCHS:
        raise RuntimeError('Önce 5. hücrede tam eğitimi bitir.')
    if (saved.get('architecture', 'early_fusion_unet') != MODEL
            or (saved.get('config') or {}).get('augment', False) != AUGMENT):
        raise RuntimeError('Tam koşudaki mimari veya veri artırma ayarı bu oturumla eşleşmiyor.')
    if torch.load(checkpoint, map_location='cpu', weights_only=True)['epoch'] != best_row(saved['history'])['epoch']:
        raise RuntimeError('En iyi model kaydını onarmak için 5. hücreyi yeniden çalıştır.')
    if not all((baseline_dir / name).is_file() for name in ('summary.json', 'per_image.csv')):
        raise FileNotFoundError(f'İlk ölçümün raporu bulunamadı: {baseline_dir}')
    baseline_summary = json.loads((baseline_dir / 'summary.json').read_text())
    if (baseline_summary['threshold_sweep']['best_threshold'] != baseline_summary['threshold']
            and not (baseline_dir / 'per_image_best.csv').is_file()):
        raise FileNotFoundError(f'Referansın seçilen eşik dosyası eksik: {baseline_dir / "per_image_best.csv"}')
    if BENCHMARK_ARCHITECTURES and not baseline_checkpoint.is_file():
        raise FileNotFoundError(f'GPU süre kıyası için ilk model kaydı gerekli: {baseline_checkpoint}')
subprocess.run([sys.executable, '-u', 'evaluate_unet.py', '--checkpoint', str(checkpoint),
                '--data-root', str(DATA_ROOT), '--threshold', '0.35', '--sweep',
                '--device', 'cuda', '--batch-size', str(BATCH_SIZE),
                '--output-dir', str(analysis_dir)], cwd=REPO_DIR, check=True,
               env={**os.environ, 'MPLBACKEND': 'Agg'})
summary = json.loads((analysis_dir / 'summary.json').read_text())
with (analysis_dir / 'threshold_sweep.csv').open(newline='') as file:
    print('Eşik  Precision   Recall       F1      IoU  Boş maske FP oranı')
    for row in csv.DictReader(file):
        scores = ' '.join(f'{float(row[key]):8.2%}' for key in ('Precision', 'Recall', 'F1', 'IoU'))
        blank_rate = f"{float(row['blank_mask_fp_rate']):.3%}" if row['blank_mask_fp_rate'] else '—'
        print(f"{float(row['threshold']):.2f}  {scores}  {blank_rate}")
print(f"En iyi val eşiği: {summary['threshold_sweep']['best_threshold']:.2f}")
print('0.35 sayımları:', summary['counts'])
if COMPARE_WITH_BASELINE:
    comparison = compare_validation(baseline_dir, analysis_dir)
    comparison['runs'] = {'baseline': BASELINE_RUN, 'candidate': RUN_NAME}
    if BENCHMARK_ARCHITECTURES:
        comparison['inference_benchmark'] = benchmark_models(
            baseline_checkpoint, checkpoint, DATA_ROOT / 'val', BATCH_SIZE, 'cuda')
    (analysis_dir / 'comparison.json').write_text(json.dumps(comparison, indent=2) + '\n', encoding='utf-8')
    for key, label in (('fixed_threshold', 'Ortak eşik'), ('best_threshold', 'Seçilen eşikler')):
        row = comparison[key]
        low, high = (value * 100 for value in row['f1_delta_ci95'])
        print(f"{label} ({row['baseline_threshold']:.2f} → {row['candidate_threshold']:.2f}) | "
              f"F1 {row['baseline_metrics']['F1']:.2%} → {row['candidate_metrics']['F1']:.2%} | "
              f"fark {row['f1_delta'] * 100:+.2f} yüzde puan | %95 aralık [{low:+.2f}, {high:+.2f}] | "
              f"piksel farkları {row['count_delta']}")
    if FULL_EPOCHS != 10:
        print('%80 doğrulama F1 hedefi:',
              'karşılandı' if summary['threshold_sweep']['best_f1'] >= 0.80 else 'karşılanmadı')
    else:
        print('Seçilen eşiklerde +1 yüzde puan hedefi:',
              'karşılandı' if comparison['best_threshold']['f1_delta'] >= 0.01 else 'karşılanmadı')
    if 'inference_benchmark' in comparison:
        benchmark = comparison['inference_benchmark']
        print(f"{benchmark['device']} | {benchmark['sample']} | {benchmark['tiles']} parça | batch {benchmark['batch_size']}")
        selected = comparison['best_threshold']
        for role, result in benchmark['models'].items():
            counts = selected[f'{role}_counts']
            print(f"{result['architecture']} | FP {counts['fp']} | FN {counts['fn']} | "
                  f"{result['parameters']:,} parametre | "
                  f"{result['median_forward_ms_per_pair']:.1f} ms/görüntü çifti")
    with (baseline_dir / 'per_image.csv').open(newline='') as old, (analysis_dir / 'per_image.csv').open(newline='') as new:
        before = {row['sample']: row for row in csv.DictReader(old)}
        after = {row['sample']: row for row in csv.DictReader(new)}
    for sample in ('val_20.png', 'val_29.png', 'val_23.png'):
        print(f"{sample} (0.35) | FP {before[sample]['fp']} → {after[sample]['fp']} | "
              f"FN {before[sample]['fn']} → {after[sample]['fn']}")
print(f'Drive çıktıları: {analysis_dir}')
for example in summary['examples'].values():
    display(Image(filename=str(analysis_dir / example['figure']), width=1000))

## 7. Nihai LEVIR-CD testi
1–2. hücreleri çalıştır ve 3. hücrede `PREPARE_TEST = True` seç. Bu hücre Drive'daki seçilmiş Early Fusion ve 30 turluk Siamese kayıtlarını açar. Üç yöntem için 128 çiftlik resmi ölçümü ve eğitim alanlarıyla kısmen örtüşen iki görüntü çıkarıldığında 126 çiftlik sonucu verir.

In [ ]:
import json
from IPython.display import Image, display

final_dir = RUNS_ROOT / 'levir_final_test'
report = final_dir / 'summary.json'
if not report.is_file():
    if not (DATA_ROOT / 'test').is_dir():
        raise RuntimeError('3. hücrede PREPARE_TEST = True seçip çalıştır.')
    subprocess.run([sys.executable, '-u', 'evaluate_test.py', '--data-root', str(DATA_ROOT),
                    '--runs-root', str(RUNS_ROOT), '--output-dir', str(final_dir),
                    '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--revision', REVISION],
                   cwd=REPO_DIR, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})
summary = json.loads(report.read_text())
method_labels = summary.get('method_labels', {})
for name, subset in summary['subsets'].items():
    label = 'Resmî test' if name == 'full' else 'Örtüşen çiftler hariç'
    print(f"\n{label} · {subset['samples']} çift")
    print(f"{'Yöntem':22} {'Eşik':>5} {'F1':>8} {'IoU':>8} {'FP':>12} {'FN':>12}")
    for method, row in subset['methods'].items():
        threshold = summary['selection'][method]['threshold']
        method_label = method_labels.get(method, method.replace('_', ' '))
        print(f"{method_label:22} {threshold:5g} {row['metrics']['F1']:8.2%} "
              f"{row['metrics']['IoU']:8.2%} {row['counts']['fp']:12,} {row['counts']['fn']:12,}")
    delta = subset['siamese_vs_early_fusion_f1_delta'] * 100
    low, high = (value * 100 for value in subset['siamese_vs_early_fusion_f1_delta_ci95'])
    print(f'Siamese – Early Fusion F1 farkı {delta:+.2f} puan, %95 aralık [{low:+.2f}, {high:+.2f}]')
print(f'\nDrive çıktıları: {final_dir}')
for example in summary['examples'].values():
    display(Image(filename=str(final_dir / example['figure']), width=1000))

## 8. S2Looking genelleme deneyi
1–2. hücreleri çalıştırıp bu hücreye geç. LEVIR-CD nihai testindeki Siamese model kaydı ve `0.50` eşiği, S2Looking'in 1.000 test çiftine doğrudan uygulanır. Yayıncı yaklaşık 10 GB'lık tek bir ZIP sunuyor. Google Drive indirme sınırına takılırsa [yayıncının arşiv sayfasındaki](https://github.com/S2Looking/Dataset) Baidu Drive bağlantısını kullanabilirsin. ZIP'i kendi Drive'ında `MyDrive/S2Looking.zip` yoluna koy veya 2. hücrede `S2LOOKING_ARCHIVE` için dosyanın yolunu yaz.

In [ ]:
import json
from IPython.display import Image, display

s2_dir = RUNS_ROOT / 's2looking_zero_shot'
s2_report = s2_dir / 'summary.json'
if not s2_report.is_file():
    for path in (RUNS_ROOT / 'levir_final_test' / 'summary.json',
                 RUNS_ROOT / 'siamese_v1_linear_e30' / 'full' / 'best.pt'):
        if not path.is_file():
            raise FileNotFoundError(f'LEVIR-CD referans dosyası bulunamadı: {path}')
    s2_archive = get_s2looking_archive()
    print('S2Looking test seti ölçülüyor.')
    subprocess.run([sys.executable, '-u', 'evaluate_s2looking.py', '--archive', str(s2_archive),
                    '--runs-root', str(RUNS_ROOT), '--output-dir', str(s2_dir),
                    '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--revision', REVISION],
                   cwd=REPO_DIR, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})
s2_summary = json.loads(s2_report.read_text())
print(f"S2Looking test | {s2_summary['samples']} çift | eşik {s2_summary['threshold']:.2f}")
for metric, value in s2_summary['metrics'].items():
    print(f'{metric:10} {value:.2%}')
low, high = s2_summary['f1_ci95']
print(f'F1 %95 aralığı [{low:.2%}, {high:.2%}]')
delta = s2_summary['f1_difference_from_levir_cd'] * 100
print(f"LEVIR-CD test F1 {s2_summary['levir_cd_test_f1']:.2%} | fark {delta:+.2f} yüzde puan")
print(f'Drive çıktıları: {s2_dir}')
for example in s2_summary['examples'].values():
    display(Image(filename=str(s2_dir / example['figure']), width=1000))

## 9. Karma eğitim için kısa prova
GPU seç ve **1–3. hücreleri** çalıştır. 3. hücrede `PREPARE_TRAIN = True` ve `PREPARE_TEST = False` kullan. Bu hücre bütün ayrımların eşleşmelerini kontrol etmek için LEVIR-CD test arşivini de hazırlar. Provada sekiz karma güncelleme yapılır ve iki doğrulama setinden birer çift ölçülür.

In [ ]:
from collections import deque
from contextlib import contextmanager
from datetime import datetime, timezone

if not all((DATA_ROOT / split).is_dir() for split in ('train', 'val')):
    raise RuntimeError('Önce 3. hücrede LEVIR-CD eğitim ve doğrulama verisini hazırla.')
levir_test = DATA_ROOT / 'test'
if not levir_test.is_dir():
    extract_split(get_archive('test', '1jj3qJD_grJlgIhUWO09zibRGJe0R4Tn0'), levir_test, 128)
mixed_archive = get_s2looking_archive()

@contextmanager
def open_mixed_log(path):
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        log = path.open('a', encoding='utf-8', buffering=1)
    except OSError as error:
        log = None
        print(f'Drive günlüğü açılamadı. Colab çıktısı devam ediyor. {error}', flush=True)
    else:
        print(f'Drive günlüğü: {path}', flush=True)

    writable = log is not None

    def append(line):
        nonlocal writable
        if writable:
            try:
                log.write(line)
            except OSError as error:
                writable = False
                print(f'Drive günlüğü yazılamadı. Colab çıktısı devam ediyor. {error}', flush=True)

    try:
        yield append
    finally:
        if log is not None:
            try:
                log.close()
            except OSError as error:
                print(f'Drive günlüğü kapatılamadı. {error}', flush=True)

def run_mixed(phase):
    command = [sys.executable, '-u', 'train_mixed_unet.py', '--phase', phase,
               '--data-root', str(DATA_ROOT), '--s2looking-archive', str(mixed_archive),
               '--runs-root', str(RUNS_ROOT), '--device', 'cuda', '--revision', REVISION]
    log_path = RUNS_ROOT / 'siamese_v1_mixed_e30' / f'{phase}.log'
    tail = deque(maxlen=12)
    with open_mixed_log(log_path) as append_log:
        append_log(f'\n{datetime.now(timezone.utc).isoformat()} | {phase} | Git {REVISION}\n')
        with subprocess.Popen(command, cwd=REPO_DIR, env={**os.environ, 'MPLBACKEND': 'Agg'},
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace') as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    tail.append(line)
                    append_log(line)
                status = process.wait()
            except BaseException:
                process.terminate()
                raise
        append_log(f'\nÇıkış kodu: {status}\n')
    if status:
        raise RuntimeError(f'Karma koşu durdu ({phase}, çıkış kodu {status}).\n'
                           f'Günlük: {log_path}\n' + ''.join(tail))

run_mixed('smoke')

## 10. Tam karma eğitim ve model seçimi
Prova tamamlanınca `RUN_MIXED_TRAIN = True` yap. S2Looking’de aynı RGB görüntüsü farklı bölümlerde kullanılmışsa test bölümüne, ardından doğrulamaya öncelik verilir. Çıkarılan çiftler ve nedenleri `split_manifest.json` dosyasına yazılır. Her batch bir LEVIR-CD ve bir S2Looking parçasıyla 106.800 güncelleme yapılır. İki doğrulama F1’i her turda ayrı ölçülür. İlerleme ve hata ayrıntıları hücrede ve Drive'daki `siamese_v1_mixed_e30/train.log` dosyasında görünür. Drive günlüğü yazılamazsa hücre çıktısı devam eder. Eğitim yarıda kalırsa aynı Git sürümü ve veriyle 1–3, 9 ve 10. hücreleri çalıştırarak son tamamlanan turdan devam et.

In [ ]:
RUN_MIXED_TRAIN = False
if RUN_MIXED_TRAIN:
    run_mixed('train')

## 11. Karma modelin test karşılaştırması
10. hücre model seçimini kaydettikten sonra çalıştır. İki model de sabit `0.50` eşiğiyle her veri setinin kendi test çiftlerinde ölçülür. Çıktılar Drive'da `change-detection-runs/siamese_v1_mixed_e30/` altında tutulur.

In [ ]:
run_mixed('test')

## Sonuçlar nerede?
30 turluk Siamese koşusunun Drive klasörü `change-detection-runs/siamese_v1_linear_e30/`. `full/history.csv` tur sonuçlarını ve öğrenme hızını, `full/last.pt` devam kaydını, `full/best.pt` ise `0.50` eşiğinde en iyi `val` F1 veren modeli tutar. 10 turluk referans ölçüm `siamese_v1_linear/val_sweep_t035/` klasöründen okunur.

6. hücre `val_sweep_t035/` altında eşik tablosunu, görüntü başına sonuçları, hata görsellerini ve `comparison.json` dosyasını kaydeder. F1 farkının aralığı, 64 doğrulama görüntüsünün eşleşmiş yeniden örneklenmesiyle hesaplanır. Bu aralık aynı iki model kaydı ve seçilmiş eşikler içindir.

7. hücre Drive'daki `change-detection-runs/levir_final_test/` klasörüne `summary.json`, `test_results.csv`, `per_image.csv` ve aynı örnekleri yan yana gösteren hata görsellerini kaydeder.

8. hücre S2Looking sonuçlarını `change-detection-runs/s2looking_zero_shot/` altında ayrı `summary.json`, `per_image.csv` ve hata görselleri olarak kaydeder.